# Stellarator mockup — results check

Plots results from the three DESC stages:
1. **Fixed-boundary equilibrium** (`eq_fixed.h5`)
2. **Coil optimisation** (`coilset.h5`) — modular coils + B·n on LCFS
3. **Free-boundary equilibrium** (`eq_free.h5`) — boundary deformation + residual B·n

Run from the `stellarator-mockup/` directory.

In [ ]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning, message=".*pynvml.*")
os.environ.setdefault("JAX_ENABLE_X64", "True")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

def show3d(fig):
    """Render a plotly figure inline without requiring nbformat or kaleido."""
    return HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

from desc.equilibrium import Equilibrium
from desc.coils import CoilSet
from desc.grid import LinearGrid
from desc.plotting import (
    plot_1d,
    plot_section,
    plot_surfaces,
    plot_3d,
    plot_coils,
    plot_boozer_surface,
    plot_comparison,
)

HERE = Path(".")

In [ ]:
eq_fixed  = Equilibrium.load(str(HERE / "eq_fixed.h5"))
coilset   = CoilSet.load(str(HERE / "coilset.h5"))
eq_free   = Equilibrium.load(str(HERE / "eq_free.h5"))

print(f"Fixed boundary : NFP={eq_fixed.NFP}, L={eq_fixed.L}, M={eq_fixed.M}, N={eq_fixed.N}")
print(f"Coilset        : {len(coilset.coils)} coils")
print(f"Free boundary  : NFP={eq_free.NFP},  L={eq_free.L}, M={eq_free.M}, N={eq_free.N}")

---
## 1 — Fixed-boundary equilibrium
### 1.1  1-D profiles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

plot_1d(eq_fixed, "iota",     ax=axes[0], color="steelblue", label="iota")
plot_1d(eq_fixed, "pressure", ax=axes[1], color="tomato",    label="pressure")
plot_1d(eq_fixed, "current",  ax=axes[2], color="seagreen",  label="current")

axes[0].set_title(r"Rotational transform $\iota(\rho)$")
axes[1].set_title(r"Pressure $p(\rho)$  [Pa]")
axes[2].set_title(r"Enclosed current $I(\rho)$  [A]")
fig.suptitle("Stage 1 — fixed-boundary equilibrium", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

plot_1d(eq_fixed, "|F|",   ax=axes[0], color="purple",     label="|F|")
plot_1d(eq_fixed, "J_phi", ax=axes[1], color="darkorange", label="J_phi")

axes[0].set_title(r"Force residual $|\mathbf{F}|(\rho)$ [N/m³]")
axes[1].set_title(r"Toroidal current density $J_\phi(\rho)$ [A/m²]")
fig.suptitle("Stage 1 — force balance", fontsize=13)
plt.tight_layout()
plt.show()

### 1.2  2-D cross-sections

In [ ]:
N_phi = 4
plot_surfaces(
    eq_fixed,
    phi=N_phi,
    rho=np.linspace(0.2, 1.0, 5),
    theta=0,
)
plt.suptitle("Stage 1 — flux surfaces (one field period)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
plot_section(eq_fixed, "|B|", log=False)
plt.suptitle("Stage 1 — |B| [T] (poloidal section)", fontsize=13)
plt.show()

plot_section(eq_fixed, "|F|_normalized", log=True)
plt.suptitle("Stage 1 — normalised |F| (log scale)", fontsize=13)
plt.show()

### 1.3  3-D view

In [ ]:
fig = plot_3d(eq_fixed, "|B|", alpha=0.8)
fig.update_layout(title="Stage 1 — fixed-boundary LCFS coloured by |B|")
show3d(fig)

---
## 2 — Coil optimisation
### 2.1  Coil geometry

In [ ]:
lengths  = np.array([c.length  for c in coilset.coils])
currents = np.array([c.current for c in coilset.coils])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(range(len(lengths)),  lengths,  color="steelblue")
axes[0].set_xlabel("coil index"); axes[0].set_ylabel("length [m]")
axes[0].set_title("Coil lengths")
axes[1].bar(range(len(currents)), currents, color="tomato")
axes[1].set_xlabel("coil index"); axes[1].set_ylabel("current [A]")
axes[1].set_title("Coil currents")
fig.suptitle("Stage 2 — optimised coil properties", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
N_phi = eq_fixed.NFP * 2
fig, axes = plot_surfaces(
    eq_fixed,
    phi=N_phi,
    rho=np.array([1.0]),
    theta=0,
)
fig.suptitle("Stage 2 — LCFS at coil toroidal angles", fontsize=13)
plt.tight_layout()
plt.show()

### 2.2  3-D — coils + plasma surface

In [ ]:
fig = plot_3d(eq_fixed, "|B|", alpha=0.4)
plot_coils(coilset, fig=fig)
fig.update_layout(title="Stage 2 — plasma surface + optimised coils")
show3d(fig)

### 2.3  B·n map on the LCFS

In [ ]:
fig = plot_3d(
    eq_fixed,
    "B*n",
    field=coilset,
    field_grid=LinearGrid(N=50, NFP=eq_fixed.NFP),
)
plot_coils(coilset, fig=fig)
fig.update_layout(title="Stage 2 — B·n on LCFS after coil optimisation [T]")
show3d(fig)

In [ ]:
from desc.objectives import QuadraticFlux

obj = QuadraticFlux(eq=eq_fixed, field=coilset, vacuum=True)
obj.build(verbose=0)
Bn_rms_stage2 = float(np.sqrt(obj.compute_scalar(*obj.xs(eq_fixed, coilset))))
print(f"B·n RMS on LCFS (stage 2, fixed boundary) : {Bn_rms_stage2:.4e} T")

---
## 3 — Free-boundary equilibrium
### 3.1  1-D profile comparison: fixed vs free

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for eq, label, color in [
    (eq_fixed, "fixed boundary", "steelblue"),
    (eq_free,  "free boundary",  "tomato"),
]:
    plot_1d(eq, "iota",     ax=axes[0], color=color, label=label)
    plot_1d(eq, "pressure", ax=axes[1], color=color, label=label)
    plot_1d(eq, "current",  ax=axes[2], color=color, label=label)

axes[0].set_title(r"$\iota(\rho)$"); axes[0].legend()
axes[1].set_title(r"$p(\rho)$ [Pa]")
axes[2].set_title(r"$I(\rho)$ [A]")
fig.suptitle("Stage 3 — profile comparison: fixed vs free boundary", fontsize=13)
plt.tight_layout()
plt.show()

### 3.2  2-D cross-section comparison

In [ ]:
N_phi = 4
phi_vals = np.linspace(0, 2 * np.pi / eq_fixed.NFP, N_phi, endpoint=False)

plot_comparison(
    [eq_fixed, eq_free],
    phi=phi_vals,
    color=["steelblue", "tomato"],
    labels=["fixed boundary", "free boundary"],
    rho=5,
    theta=0,
)
plt.suptitle("Stage 3 — LCFS comparison: fixed vs free boundary", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_section(eq_fixed, "|B|", ax=axes[0], log=False)
axes[0].set_title("Fixed boundary — |B| [T]")

plot_section(eq_free, "|B|", ax=axes[1], log=False)
axes[1].set_title("Free boundary — |B| [T]")

fig.suptitle("Stage 3 — |B| cross-section comparison", fontsize=13)
plt.tight_layout()
plt.show()

### 3.3  3-D — free-boundary LCFS + coils

In [ ]:
fig = plot_3d(eq_free, "|B|", alpha=0.6)
plot_coils(coilset, fig=fig)
fig.update_layout(title="Stage 3 — free-boundary LCFS coloured by |B| + coils")
show3d(fig)

### 3.4  Residual B·n on the free-boundary LCFS

In [ ]:
fig = plot_3d(
    eq_free,
    "B*n",
    field=coilset,
    field_grid=LinearGrid(N=50, NFP=eq_free.NFP),
)
plot_coils(coilset, fig=fig)
fig.update_layout(title="Stage 3 — residual B·n on free-boundary LCFS [T]")
show3d(fig)

In [ ]:
obj_free = QuadraticFlux(eq=eq_free, field=coilset, vacuum=True)
obj_free.build(verbose=0)
Bn_rms_stage3 = float(np.sqrt(obj_free.compute_scalar(*obj_free.xs(eq_free, coilset))))

print(f"B·n RMS — stage 2 (fixed boundary) : {Bn_rms_stage2:.4e} T")
print(f"B·n RMS — stage 3 (free  boundary) : {Bn_rms_stage3:.4e} T")
print(f"Reduction factor                   : {Bn_rms_stage2 / Bn_rms_stage3:.2f}×")

### 3.5  Boozer spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_boozer_surface(eq_fixed, ax=axes[0])
axes[0].set_title("Fixed boundary — Boozer |B| spectrum")

plot_boozer_surface(eq_free, ax=axes[1])
axes[1].set_title("Free boundary — Boozer |B| spectrum")

fig.suptitle("Stage 3 — Boozer spectrum comparison", fontsize=13)
plt.tight_layout()
plt.show()

### 3.6  Force balance on the free-boundary equilibrium

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_section(eq_free, "|F|_normalized", ax=axes[0], log=True)
axes[0].set_title("Free boundary — normalised |F| (log)")

plot_1d(eq_free, "|F|", ax=axes[1], color="purple")
axes[1].set_title(r"Free boundary — $|\mathbf{F}|(\rho)$ [N/m³]")

fig.suptitle("Stage 3 — force balance residual", fontsize=13)
plt.tight_layout()
plt.show()